# ChemBounce 1.x — stable v1.0.2
<a href="https://colab.research.google.com/github/jyryu3161/chembounce/blob/v1.0.2/chembounce_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run **Checkout**, **Environment and reference data**, **Basic input parameters**, then **Run**.
All optional cells can be skipped. A CPU runtime is sufficient.

If you used an older notebook that installed Anaconda into `/usr/local`, choose
**Runtime → Disconnect and delete runtime** before starting this notebook.
Setup installs an isolated Python 3.10 environment and downloads the reference library;
the first setup can take several minutes. Do not continue after a red error.
A failed calculation does not produce a downloadable result archive.

In [1]:
#@title Checkout
import os
import subprocess
import sys
from pathlib import Path

RELEASE = "v1.0.2"
checkout = Path("/content") / f"chembounce-{RELEASE}"
if not checkout.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RELEASE,
                    "https://github.com/jyryu3161/chembounce.git", str(checkout)], check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
release_commit = subprocess.check_output(
    ["git", "rev-parse", f"refs/tags/{RELEASE}^{{commit}}"], cwd=checkout, text=True).strip()
if head != release_commit:
    raise RuntimeError(f"Checkout is not {RELEASE}. Start a fresh runtime.")
subprocess.run(["git", "diff", "--exit-code", "HEAD"], cwd=checkout, check=True)
os.chdir(checkout)
sys.path.insert(0, str(checkout))
from colab_support import DEFAULTS, ensure_data, run_and_archive, setup_environment
from google.colab import files
print(f"Using {RELEASE} ({head[:12]})")

Using v1.0.2 (3739653d77f5)


In [2]:
#@title Environment and reference data
#@markdown Run this cell again to repair an interrupted package installation.
bootstrap = Path("/content/chembounce-bootstrap-1x")
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--target", str(bootstrap), "--upgrade", "uv==0.9.0"], check=True)
chembounce_python = setup_environment(
    checkout, Path("/content") / f"chembounce-env-{RELEASE}", bootstrap / "bin/uv")
ensure_data(checkout, chembounce_python)

# Initialize every optional setting, including when optional cells are skipped.
for name, value in DEFAULTS.items():
    globals()[name] = value.copy() if isinstance(value, list) else value
archive = None
print("Setup complete. Continue with Basic input parameters and Run.")

RuntimeError: ChemBounce environment check failed:
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/chembounce-v1.0.2/chembounce.py", line 22, in <module>
    import scaffoldgraph_fragmenter as sg
  File "/content/chembounce-v1.0.2/scaffoldgraph_fragmenter.py", line 20, in <module>
    from scaffoldgraph import get_murcko_scaffold
  File "/content/chembounce-env-v1.0.2/lib/python3.10/site-packages/scaffoldgraph/__init__.py", line 9, in <module>
    from . import vis
  File "/content/chembounce-env-v1.0.2/lib/python3.10/site-packages/scaffoldgraph/vis/__init__.py", line 5, in <module>
    from .utils import (
  File "/content/chembounce-env-v1.0.2/lib/python3.10/site-packages/scaffoldgraph/vis/utils.py", line 5, in <module>
    import matplotlib.pyplot as plt
  File "/content/chembounce-env-v1.0.2/lib/python3.10/site-packages/matplotlib/__init__.py", line 1299, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
  File "/content/c

In [ ]:
#@title Basic input parameters
#@markdown ##Structures
#@markdown ####input_SMILES
#@markdown #####Input SMILES, the target molecular structure
# imatinib: CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5
# Gefitinib: COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4
# Losartan: CCCCC1=NC(=C(N1CC2=CC=C(C=C2)C3=CC=CC=C3C4=NNN=N4)CO)Cl
input_smiles="NC1=NC(N(C2=NC(C(F)(F)F)=CC=C21)C3=C(Cl)C=CC=C3)=O" #@param {type:'string'}

#@markdown ####core_smiles (optional)
#@markdown #####(Optional) Core SMILES which should not be altered while scaffold hopping
core_smiles="" #@param {type:'string'}

#@markdown ##Parameters
#@markdown ####frag_max_n
#@markdown #####Maximal number of scaffold-hopped candidates for a fragment
frag_max_n=100 #@param {type:'integer'}

#@markdown ####tanimoto_threshold
#@markdown #####Tanimoto similarity threshold, between 0 and 1: used to exclude irrelated molecular structure, based on the similarity between the original structure and scaffold-hopped one. Default is 0.5
tanimoto_threshold=0.5 #@param {type:'slider',min:0.0, max:1.0, step:0.01}

#@markdown ####output_dir
#@markdown #####Output location
output_dir = './output_Gefitinib' #@param {type:'string'}

#@markdown Inspect filter diagnostics when no candidates pass; adjust limits explicitly for your application.
#@markdown #### Lipinski's rule of Five
#@markdown ##### Application of Lipinski\'s rule of five :
#@markdown ##### logp_max=5, mw_max=500, h_donor_max=5, h_acceptor_max=10
lipinski=True #@param {type:'boolean'}

print(
    f"Input SMILES\t:{input_smiles}",
    f"\nCore SMILES\t:{core_smiles}",
    f"\nTanimoto threshold\t:{tanimoto_threshold}",
    f"\nTop fragments\t:{frag_max_n}",
    f"\nLipinski\'s rule of five\t:{lipinski}",
    f"\nOutput location\t:{os.path.abspath(output_dir)}"
)

## Optional parameters
Skip these cells to use the defaults. Run a cell only to change its settings, then run **Run** again.
Existing outputs are preserved; a repeated calculation uses a new output directory.

Custom libraries must contain one scaffold SMILES per line. Fragment SMILES select
substructures to replace; optional replacement files can supply scaffold lists for them.

In [ ]:
#@title (Optional) Custom library
use_custom_library = False #@param {type:'boolean'}
custom_lib_file_name = "custom_scaffolds.txt" #@param {type:'string'}
if use_custom_library:
    files.upload()
    library = (checkout / custom_lib_file_name).resolve()
    if not library.is_file():
        raise FileNotFoundError(f"Upload the library and set its exact name: {library}")
    subprocess.run([str(chembounce_python), str(checkout / "generate_fingerprints.py"),
                    "--input", str(library), "--output", str(library) + ".npz",
                    "--fp-size", "2048", "--radius", "2", "--batch-size", "10000",
                    "--verify"], cwd=checkout, check=True)

In [ ]:
#@title (Optional) Maximal iteration and number of candidates
#@markdown Leave None to use the 1.x defaults. Positive integers only.
overall_max_n = None #@param {type:'integer'}
scaffold_top_n = None #@param {type:'integer'}
cand_max_n__rplc = 10 #@param {type:'integer'}

In [ ]:
#@title (Optional) Threshold options

#@markdown #### Molecular properties
#@markdown ##### Min/max for QED, SAscore, logP, MW, number of H donors and number of H acceptors
#@markdown ##### User-defined options take precedence over Lipinski's rule.
qed_min=None #@param {type:'number'}
qed_max=None #@param {type:'number'}
sa_min=None #@param {type:'number'}
sa_max=None #@param {type:'number'}
logp_min=None #@param {type:'number'}
logp_max=None #@param {type:'number'}
mw_min=None #@param {type:'number'}
mw_max=None #@param {type:'number'}
h_donor_min=None #@param {type:'number'}
h_donor_max=None #@param {type:'number'}
h_acceptor_min=None #@param {type:'number'}
h_acceptor_max=None #@param {type:'number'}

In [ ]:
#@title (Optional) Fragments and replacement scaffolds
#@markdown Comma-separated fragment SMILES. Leave empty for automatic fragmentation.
fragment_smiles = "" #@param {type:'string'}
fragments = [value.strip() for value in fragment_smiles.split(",") if value.strip()]
#@markdown Optional uploaded replacement files, in the same order as fragments.
#@markdown Leave all empty to search the default library, or use empty entries for selected fragments.
replace_scaffold_file = "" #@param {type:'string'}
replace_scaffold_files = ([value.strip() for value in replace_scaffold_file.split(",")]
                          if replace_scaffold_file else [])
if replace_scaffold_files and len(replace_scaffold_files) != len(fragments):
    raise ValueError("Use one replacement-file entry per fragment; empty entries use the default library.")
print(f"User-specified fragments: {len(fragments)}")

In [ ]:
#@title Run
#@markdown Results are downloaded only after the process and result files pass validation.
#@markdown Zero candidates is reported explicitly, with filter diagnostics in run_summary.json.
#@markdown On error, inspect run.log and notebook_run.json in the printed output directory.
archive = None
settings = {name: globals()[name] for name in DEFAULTS}
settings["input_smiles"] = input_smiles
archive = run_and_archive(chembounce_python, checkout, output_dir, settings)
files.download(str(archive))